### COMPARISON WITH THE BASELINE EXPERIMENTS OF : Differentially private sliced wasserstein distance (https://arxiv.org/abs/2107.01848)

**Import the required libraries: we are using torch instead of jax for the comparison**

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

**Check if CUDA is available**

In [11]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


**Functions required for the comparison**

In [12]:
def generate_data(n):
    X = torch.randn(n, 2).to(device)  
    theta = 2 * np.pi * torch.rand(n).to(device)  
    Z = torch.stack((0.75 * torch.cos(theta), 0.75 * torch.sin(theta)), dim=1).to(device)  
    return X, Z

class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
    
    def forward(self, x):
        return self.model(x)

def plot_points(Z, X_transformed, epoch, loss_target):
    plt.figure(figsize=(8, 8))
    plt.scatter(X_transformed[:, 0].detach().cpu().numpy(), X_transformed[:, 1].detach().cpu().numpy(), label='Transformed X', alpha=0.05)
    plt.scatter(Z[:, 0].cpu().numpy(), Z[:, 1].cpu().numpy(), label='Z (Target)', alpha=0.5)
    #plt.legend()
    plt.title(r"Epoch {epoch}, loss = {loss}, $\epsilon_X = \infty$, $\epsilon_Z = {epsilon}$".format(epoch=epoch, loss=loss_target, epsilon=epsilon))
    plt.xlim(-1, 1)
    plt.ylim(-1, 1)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.show()

def true_loss(X_transformed, Z, U):
    X_proj = torch.matmul(X_transformed, U)
    Z_proj = torch.matmul(Z, U)

    X_proj_sorted, _ = torch.sort(X_proj, dim=0)
    Z_proj_sorted, _ = torch.sort(Z_proj, dim=0)
    
    return torch.mean((X_proj_sorted - Z_proj_sorted) ** 2)

def sliced_w2_loss(X_transformed, Z_proj_batch, U):
    X_proj = torch.matmul(X_transformed, U)
    
    X_proj_sorted, _ = torch.sort(X_proj, dim=0)
    Z_proj_batch_sorted, _ = torch.sort(Z_proj_batch, dim=0)

    return torch.mean((X_proj_sorted - Z_proj_batch_sorted) ** 2)

**Set the parameters of the experiment**

In [ ]:
d = 2
k = 1

sensitivity = np.sqrt(2)



n = 100000
n_prime=10000 #Size of the subsampled datased for privacy amplification by subsampling
p=n_prime/n #subsampling rate

batch_size = n_prime

learning_rate = 0.0075
epochs = 100001

X, Z = generate_data(n)

model = NeuralNet().to(device)  
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

subsampled_idx = np.random.permutation(n)[0:n_prime]

m = 15
U = 1 / np.sqrt(d) * torch.randn(d, m).to(device)  

Z_proj = torch.matmul(Z, U)
privacy_noise = 0.7
Z_proj += privacy_noise * sensitivity * torch.randn(Z_proj.shape).to(device)  

delta = 0.1 / n
m_prime = m * k

alpha_max = privacy_noise * np.sqrt(d) + 1
alphas_list = np.linspace(1, alpha_max, 1000)
epsilon_subsampled = 1e6

for alpha in alphas_list:
    gamma = privacy_noise**(-2) * (alpha**2 - alpha)
    if gamma < d:
        epsilon_subsampled = min(epsilon_subsampled, m_prime * alpha / (2 * privacy_noise**2 * (d-gamma)) + np.log(p / delta) / (alpha - 1))

print(epsilon_subsampled)
epsilon = np.log1p(p * (np.expm1(epsilon_subsampled)))
print(epsilon)

**Training loop**

In [ ]:
for epoch in range(epochs):
    idx = torch.randint(0, n_prime, (batch_size,)).to(device)  
    X_batch, Z_proj_batch = X[subsampled_idx][idx], Z_proj[subsampled_idx][idx]
    optimizer.zero_grad()
    X_transformed = model(X_batch)
    loss = sliced_w2_loss(X_transformed + privacy_noise * sensitivity * torch.randn(X_transformed.shape).to(device)  , Z_proj_batch, U)
    loss.backward()
    
    optimizer.step()
    
    if epoch % 1000 == 0:
        X_transformed = model(X)
        loss_target = true_loss(X_transformed, Z, U)
        print(f"Epoch {epoch}: Loss = {loss.item()}")
        if epoch % 25000 == 0:
            plot_points(Z, X_transformed, epoch, loss_target)